In [1]:
import sys, os

%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


# Imports

In [2]:
%%RecordEvent
import numpy as np
import os
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import time

In [3]:
import time

start_time = time.perf_counter()

# Load Data

In [4]:
### cell 0 ###

train_df = pd.read_csv(
    "/home/dias-benchmarks/notebooks/lextoumbourou/feedback3-eda-hf-custom-trainer-sift/input/feedback-prize-english-language-learning/train.csv"
)
test_df = pd.read_csv(
    "/home/dias-benchmarks/notebooks/lextoumbourou/feedback3-eda-hf-custom-trainer-sift/input/feedback-prize-english-language-learning/test.csv"
)

# -- STEFANOS -- Replicate Data

In [5]:
### cell 1 ###

factor = 200
if (
    "IREWR_LESS_REPLICATION" in os.environ
    and os.environ["IREWR_LESS_REPLICATION"] == "True"
):
    factor = factor // 5
train_df = pd.concat([train_df] * factor)
test_df = pd.concat([test_df] * factor)
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 782200 entries, 0 to 3910
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   text_id      782200 non-null  object 
 1   full_text    782200 non-null  object 
 2   cohesion     782200 non-null  float64
 3   syntax       782200 non-null  float64
 4   vocabulary   782200 non-null  float64
 5   phraseology  782200 non-null  float64
 6   grammar      782200 non-null  float64
 7   conventions  782200 non-null  float64
dtypes: float64(6), object(2)
memory usage: 53.7+ MB


Let's see a row from each dataset.

In [6]:
### cell 2 ###

train_df.head()

,text_id,full_text,cohesion,syntax,vocabulary,phraseology,grammar,conventions
0,0016926B079C,I think that students would benefit from learn...,3.5,3.5,3.0,3.0,4.0,3.0
1,0022683E9EA5,When a problem is a change you have to let it ...,2.5,2.5,3.0,2.0,2.0,2.5
2,00299B378633,"Dear, Principal\n\nIf u change the school poli...",3.0,3.5,3.0,3.0,3.0,2.5
3,003885A45F42,The best time in life is when you become yours...,4.5,4.5,4.5,4.5,4.0,5.0
4,0049B1DF5CCC,Small act of kindness can impact in other peop...,2.5,3.0,3.0,3.0,2.5,2.5


In [7]:
### cell 3 ###

test_df.head()

,text_id,full_text
0,0000C359D63E,when a person has no experience on a job their...
1,000BAD50D026,Do you think students would benefit from being...
2,00367BB2546B,"Thomas Jefferson once states that ""it is wonde..."
0,0000C359D63E,when a person has no experience on a job their...
1,000BAD50D026,Do you think students would benefit from being...


Then the size of each dataset.

In [8]:
### cell 4 ###

len(train_df), len(test_df)

(782200, 600)

In [9]:
### cell 5 ###

LABEL_COLUMNS = [
    "cohesion",
    "syntax",
    "vocabulary",
    "phraseology",
    "grammar",
    "conventions",
]

# Text Examples

## Random Examples

In [10]:
### cell 6 ###

texts = train_df.sample(frac=1, random_state=420).head(4)

## Lowest Scoring Examples

In [11]:
### cell 7 ###

train_df["total_score"] = train_df[LABEL_COLUMNS].sum(axis=1)
lowest_df = train_df.sort_values("total_score").head(4)

## Highest Scoring Examples

In [12]:
### cell 8 ###

train_df["total_score"] = train_df[LABEL_COLUMNS].sum(axis=1)
highest_df = train_df.sort_values("total_score", ascending=False).head(4)

# Text Overview

## Word Count

In [13]:
### cell 9 ###


train_df["word_count"] = train_df.full_text.apply(lambda x: len(x.split()))

Mean word count:

In [14]:
### cell 10 ###

train_df["word_count"].mean()

np.float64(430.4929685502429)

Max word count:

In [15]:
### cell 11 ###

train_df["word_count"].max()

np.int64(1260)

In [16]:
end_time = time.perf_counter()
print(end_time - start_time)

16.47233308200157
